# Pesquisa Nacional por Amostra de Domicílios - PNAD COVID19

Objetiva estimar o número de pessoas com sintomas referidos associados à síndrome gripal e monitorar os impactos da pandemia da COVID-19 no mercado de trabalho brasileiro.

A coleta da Pesquisa Nacional por Amostra de Domicílios - PNAD COVID19 teve início em 4 de maio de 2020, com entrevistas realizadas por telefone em, aproximadamente, 48 mil domicílios por semana, totalizando cerca de 193 mil domicílios por mês, em todo o Território Nacional. A amostra é fixa, ou seja, os domicílios entrevistados no primeiro mês de coleta de dados permanecerão na amostra nos meses subsequentes, até o fim da pesquisa.

O questionário se divide em duas partes, sendo uma direcionada a questões de saúde, especificamente sobre sintomas associados à síndrome gripal e outra, a questões de trabalho. Nas questões de saúde, investiga-se a ocorrência de alguns dos principais sintomas da COVID19 no período de referência da pesquisa, considerando-se todos os moradores do domicílio. Para aqueles que apresentaram algum sintoma, perguntam-se quais as providências tomadas para alivio dos sintomas; se buscaram por atendimento médico devido a esses sintomas; e o tipo de estabelecimento de saúde procurado. Nas questões de trabalho, busca-se classificar a população em idade de trabalhar nas seguintes categorias: ocupados, desocupados e pessoas fora da força de trabalho. Investiga-se, ainda, os seguintes aspectos: ocupação e atividade; afastamento do trabalho e o motivo do afastamento; exercício de trabalho remoto; busca por trabalho; motivo por não ter procurado trabalho; horas semanais efetivamente e habitualmente trabalhadas; assim como o rendimento efetivo e habitual do trabalho. Por fim, visando compor o rendimento domiciliar, pergunta-se se algum morador recebeu outros rendimentos não oriundos do trabalho, tais como: aposentadoria, BPC-LOAS, Bolsa Família, algum auxílio emergencial relacionado à COVID, seguro desemprego, aluguel e outros. Cabe ressaltar que a PNAD COVID19 é uma pesquisa com instrumento dinâmico de coleta das informações; portanto, o questionário está sujeito a alterações ao longo do período de sua aplicação.

A pesquisa prevê divulgações semanais para alguns indicadores, em nível Brasil, e divulgações mensais para um conjunto mais amplo de indicadores, por Unidades da Federação.

Os resultados da PNAD COVID19 são pioneiros no sentido de constituírem a primeira divulgação de Estatísticas Experimentais elaboradas pelo IBGE, as quais estão alinhadas com a estratégia de modernização do Instituto e permitem a ampliação das ofertas de informação para atender às necessidades de seus usuários.

https://www.ibge.gov.br/estatisticas/downloads-estatisticas.html?caminho=Trabalho_e_Rendimento/Pesquisa_Nacional_por_Amostra_de_Domicilios_PNAD_COVID19/Microdados/Dados

# Características dos dados

Parte 1 - Identificação e Controle
Parte A - Características gerais dos moradores
Parte B - COVID19 - Todos os moradores
Parte C - Características de trabalho das pessoas de 14 anos ou mais de idade
Parte D - Rendimento de outras fontes dos moradores de 14 anos ou mais de idade
Parte E - Empréstimos
Parte Suplementar 01 - Características da habitação


# Prospecção de Internação e sedação, entubação ou colocado em respiraçãoartificial com ventilador de COVID-19 em relação aos sintomas sentidos ediagnóstico médico de alguma comorbidade


A COVID-19 afeta diferentes pessoas de diferentes maneiras. A maioria das pessoas infectadas apresentará sintomas leves a moderados da doença e não precisarão ser hospitalizadas. De acordo com a base de dados coletada e tratada, a nossa intenção é predizer qual é a chance de uma pessoa que sentiu algum dos sintomas (febre, tosse, dor de garganta, difilcudade de respirar dor de cabeça, dor no peito, nausea, nariz entupido ou escorrendo, fadiga, dor nos olhos, perdade de paladar ou olfato, dor muscular ou diarreia) e que tem ou não algum diagnostico de algum comorbidade como diabetes, obesidade, hipertensão, tuberculose, entre outros, aumentam a possíbilidade de internação ou entubação.

In [21]:
import cx_Oracle
import csv
import matplotlib as mpl
import pandas as pd
import mpl_toolkits.mplot3d
from scipy.interpolate import *
import matplotlib.pyplot as plt
from sklearn import linear_model
import tkinter as tk
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import seaborn
import numpy as np
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import confusion_matrix
from sklearn.cluster import KMeans
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.model_selection import KFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [22]:
result = pd.read_csv(r"C:\Users\et01133\Desktop\Mestrado\covid_estabelecimento.csv")

In [23]:
#result
#result.columns
#result.head

In [24]:
NUMBER_OF_CLUSTERS = 4
kmeans = KMeans(n_clusters = NUMBER_OF_CLUSTERS, init = 'k-means++', max_iter = 300, n_init = 10)

# Tratamento dos dados

# Passo 1

Vamos remover as colunas que julgamos não acrescentarem nenhum valor ao nosso modelo.


    NUMERO            - Número de Ordem
    SEMANA            - Semana no mês
    NUMERO_ENTREVISTA - Número da Entrevista
    NUMERO_ORDEM      - Número de Ordem
    UF                - Unidade Federativa
    SITUACAO          - Situação do domicílio (Urbana; Rural)
    DOMICILIO         - 
    RESPONDEU         - Quem respondeu ao questionário
    ESCOLARIDADE      - Nível escolar
 
# Passo 2

Para predizer se uma pessoa tem a possibilidade de ser internada vamos revomer a váriavel B006 - Durante a internação, foi sedado, entubado e colocado em respiração artificial com ventilador?


In [25]:
#Removendo as colunas
result.drop(["NUMERO","SEMANA","NUMERO_ENTREVISTA","NUMERO_ORDEM","UF","SITUACAO","DOMICILIO","RESPONDEU","ESCOLARIDADE"], axis="columns", inplace=True)
result.drop(["B006"], axis="columns", inplace=True)

In [26]:
#result.columns
#result['B005'].value_counts()
#result['B006'].value_counts()
#result.isna().sum()

In [27]:
from sklearn.model_selection import train_test_split

X = result.drop(['B005'], axis = "columns")
y = result.B005

In [28]:

column_names_train = result.drop(['B005'], axis = "columns").columns
column_name_target = result[['B005']].columns

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 13)

In [29]:
# Cria nosso pipeline para pré-processamento com imputação, dummização e normalização
encoder_imputer_scaled_transformer = make_column_transformer(
    (make_pipeline(
        (KNNImputer(n_neighbors = 3)),
        (StandardScaler())
    ) , make_column_selector(dtype_include = np.number)),
    (make_pipeline(
        SimpleImputer(strategy = 'most_frequent'),
        OneHotEncoder(handle_unknown = 'ignore'),
    ), make_column_selector(dtype_exclude = np.number)),
    remainder = 'passthrough'
)

encoder_imputer_scaled_transformer.fit(X_train)

X_train_processed = encoder_imputer_scaled_transformer.transform(X_train)
X_test_processed = encoder_imputer_scaled_transformer.transform(X_test)

# Treina o algoritmo knn com k = 3
knn = KNeighborsClassifier(n_neighbors = 3)

knn.fit(X_train_processed, y_train)

# Avalia a performance do algoritmo utilizando a métrica de accuracy
y_pred = knn.predict(X_test_processed)

accuracy = accuracy_score(y_true = y_test, 
                          y_pred = y_pred)

accuracy

0.4743083003952569

## Aplicando o $k$-fold cross-validation para avaliação do modelo

**O primeiro ponto importante é que, como a rotina de $k$-fold cross-validation já cria folds disjuntos para treinamento e teste, não há mais a necessidade de se realizar um holdout. Ou seja, usamos toda a base de dados.**

Vamos aplicar os passos de pré-processamento em toda base, tal como fizemos para os conjuntos de treinamento e de teste acima.

In [30]:
X_processed = encoder_imputer_scaled_transformer.fit_transform(X)

Vamos aproveitar todo o código, mas agora vamos rodá-lo com um $k$-fold cross-validation com $k=10$. 

A função `cross_val_score` do subpacote `sklearn.model_selection` realiza o $k$-fold cross-validation de forma automática. 
Basta passarmos os folds no parâmetro `cv` e a métrica de avaliação no parâmetro `scoring`. 
Lembrar que devemos treinar usando o conjunto de treinamento já pré-processado.

In [31]:
from sklearn.model_selection import cross_val_score

folds = KFold(n_splits=10, shuffle=True).split(X_processed)

scores = cross_val_score(estimator = knn, 
                         X = X_processed, # usamos a base completa agora
                         y = y, 
                         cv = folds, 
                         scoring = 'accuracy')

print("Acurácia em cada fold: {}".format(scores))
print("Acurácia média nos folds: {}".format(np.mean(scores)))

Acurácia em cada fold: [0.46706192 0.49736495 0.49209486 0.46969697 0.47299078 0.49538867
 0.50263505 0.48418972 0.49604743 0.49341238]
Acurácia média nos folds: 0.4870882740447957


In [36]:
from sklearn.model_selection import GridSearchCV

grid_valores = {"n_neighbors": range(1, 31)}

knn_automatico = GridSearchCV(estimator = KNeighborsClassifier(), 
                              param_grid = grid_valores, 
                              cv = 10,
                              scoring = 'accuracy',
                              refit = True)

knn_automatico.fit(X_train_processed, y_train)

print("Melhor parâmetro do modelo knn: ")
print(knn_automatico.best_params_)

print("Desempenho médio no fold de validação: ")
print(np.mean(knn_automatico.cv_results_['mean_test_score']))

y_pred = knn_automatico.predict(X_test_processed)

accuracy = accuracy_score(y_true = y_test, 
                          y_pred = y_pred)

print("Desempenho médio no fold de teste: ")
accuracy


C:\ProgramData\Anaconda3\lib\site-packages\sklearn\model_selection\_split.py:667: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=10.
  % (min_groups, self.n_splits)), UserWarning)


KeyboardInterrupt: 

In [35]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("preprocessamento", encoder_imputer_scaled_transformer),
    ("knn", KNeighborsClassifier())
])

grid_valores = {"knn__n_neighbors": range(1, 31)}

classificador = GridSearchCV(estimator = pipeline, 
                              param_grid = grid_valores,
                              cv = 10, 
                              scoring = "accuracy", 
                              refit = True)

classificador

NameError: name 'GridSearchCV' is not defined

In [34]:
from sklearn.model_selection import StratifiedKFold

pipeline_estendido = Pipeline([
    ("pre_process", make_column_transformer(
                              (Pipeline([
                                  ('imputer', KNNImputer(n_neighbors = 3)),
                                  ('scaler', StandardScaler())
                              ]) , make_column_selector(dtype_include = np.number)),
                              (Pipeline([
                                  ("imputer", SimpleImputer(strategy = 'most_frequent', fill_value = "unknown")),
                                  ("encoder", OneHotEncoder(handle_unknown = 'ignore'))
                              ]), make_column_selector(dtype_exclude = np.number))
                          )),
    ("knn", KNeighborsClassifier())                           
])

param_grid_estendido = {
    "pre_process__pipeline-1__imputer__n_neighbors": range(1, 3),
    "pre_process__pipeline-1__scaler": ["passthrough", StandardScaler()], # o passo de normalização torna-se opcional
    "knn__n_neighbors": range(5, 10)
}

# Quantidade de folds no outer loop
outer_folds_cv = StratifiedKFold(n_splits= 5, shuffle = True)

# Quantidade de folds no inner loop
inner_folds_cv = StratifiedKFold(n_splits = 5, shuffle = True)

classifier_nested_cv = GridSearchCV(estimator=pipeline_estendido, param_grid=param_grid_estendido, cv=inner_folds_cv)

# Rodo na base toda, sem precisar fazer uma pré-divisão com holdout
nested_score = cross_val_score(classifier_nested_cv, X=X, y=y, cv=outer_folds_cv)

NameError: name 'Pipeline' is not defined

In [33]:
from sklearn.metrics import plot_confusion_matrix

plot_confusion_matrix(classificador_estendido, X_test, y_test, values_format = '')

NameError: name 'classificador_estendido' is not defined

In [32]:
import seaborn
import matplotlib.pyplot as plt

results_cross_validation = []
for k in range(3, 51):
  for time in range(30):
    kf = KFold(n_splits=k, shuffle=True).split(X_processed)
    scores = cross_val_score(knn, X_processed, y, cv=kf, scoring='accuracy')
    results_cross_validation = results_cross_validation + [[k, time, np.mean(scores)]]

KeyboardInterrupt: 

In [ ]:
df = pd.DataFrame(results_cross_validation,columns=['k','time_id', 'accuracy'])

seaborn.relplot(data = df,
                x = 'k',
                y = 'accuracy',
                kind = 'line')
plt.show()
